In [1]:
import sys
sys.path.insert(0, '../src')
from database import load_data_to_db, run_query, get_summary_stats

# ── Step 1: Load data into DB (takes ~2 mins) ─────────────────────
load_data_to_db("../data/cleaned/traffic_clean.csv")

⏳ Loading data into SQLite...
✅ Connected to database: ../data/traffic_violations.db
✅ Loaded 2,070,115 rows into 'violations' table
✅ Indexes created for fast querying


True

In [2]:
import importlib, sys

# Clear cache
for mod in list(sys.modules.keys()):
    if 'database' in mod:
        del sys.modules[mod]

sys.path.insert(0, '../src')
import database
importlib.reload(database)
from database import load_data_to_db, run_query, get_summary_stats

# Data is already in DB, just recreate indexes
conn = database.create_connection()
cursor = conn.cursor()

indexes = [
    "CREATE INDEX IF NOT EXISTS idx_date      ON violations('Date Of Stop')",
    "CREATE INDEX IF NOT EXISTS idx_violation  ON violations(ViolationCategory)",
    "CREATE INDEX IF NOT EXISTS idx_gender     ON violations(Gender)",
    "CREATE INDEX IF NOT EXISTS idx_race       ON violations(Race)",
    "CREATE INDEX IF NOT EXISTS idx_make       ON violations(Make)",
    "CREATE INDEX IF NOT EXISTS idx_accident   ON violations(Accident)",
    "CREATE INDEX IF NOT EXISTS idx_subagency  ON violations(SubAgency)",
]

for idx in indexes:
    cursor.execute(idx)

conn.commit()
conn.close()
print("✅ All indexes created successfully!")

✅ Connected to database: ../data/traffic_violations.db
✅ All indexes created successfully!


In [4]:
import sqlite3
import pandas as pd
import os

# ── Fix: use correct DB path ──────────────────────────────────────
DB_PATH = "../data/traffic_violations.db"  # no 's' — matches what was created

conn = sqlite3.connect(DB_PATH)

# Confirm table exists
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in DB:", tables['name'].tolist())

# Create indexes
cursor = conn.cursor()
indexes = [
    "CREATE INDEX IF NOT EXISTS idx_violation ON violations(ViolationCategory)",
    "CREATE INDEX IF NOT EXISTS idx_gender    ON violations(Gender)",
    "CREATE INDEX IF NOT EXISTS idx_race      ON violations(Race)",
    "CREATE INDEX IF NOT EXISTS idx_make      ON violations(Make)",
    "CREATE INDEX IF NOT EXISTS idx_accident  ON violations(Accident)",
    "CREATE INDEX IF NOT EXISTS idx_subagency ON violations(SubAgency)",
]
for idx in indexes:
    cursor.execute(idx)

conn.commit()
print("✅ Indexes created!")

# ── Run all 5 queries ─────────────────────────────────────────────
print("\n📊 Query 1 — Top Violation Types:")
q1 = pd.read_sql("""
    SELECT ViolationCategory,
           COUNT(*) as total,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM violations), 2) as pct
    FROM violations
    GROUP BY ViolationCategory
    ORDER BY total DESC
""", conn)
print(q1.to_string(index=False))

print("\n🚨 Query 2 — Districts with Highest Accident Rates:")
q2 = pd.read_sql("""
    SELECT SubAgency,
           COUNT(*) as total_stops,
           SUM(CASE WHEN Accident = 1 THEN 1 ELSE 0 END) as accidents,
           ROUND(SUM(CASE WHEN Accident = 1 THEN 1 ELSE 0 END)*100.0/COUNT(*),2) as accident_rate
    FROM violations
    WHERE SubAgency != 'nan'
    GROUP BY SubAgency
    ORDER BY accident_rate DESC
    LIMIT 10
""", conn)
print(q2.to_string(index=False))

print("\n🔁 Query 3 — Repeat Offenders:")
q3 = pd.read_sql("""
    SELECT SeqID,
           COUNT(*) as charge_count,
           MAX(ViolationCategory) as main_violation
    FROM violations
    GROUP BY SeqID
    HAVING charge_count > 3
    ORDER BY charge_count DESC
    LIMIT 10
""", conn)
print(q3.to_string(index=False))

print("\n🚗 Query 4 — Vehicle Makes with Highest Accident Rates:")
q4 = pd.read_sql("""
    SELECT Make,
           COUNT(*) as total,
           SUM(CASE WHEN Accident = 1 THEN 1 ELSE 0 END) as accidents,
           ROUND(SUM(CASE WHEN Accident=1 THEN 1 ELSE 0 END)*100.0/COUNT(*),2) as acc_rate
    FROM violations
    WHERE Make NOT IN ('nan','N/A','')
    GROUP BY Make
    HAVING total > 1000
    ORDER BY acc_rate DESC
    LIMIT 10
""", conn)
print(q4.to_string(index=False))

print("\n" + "="*45)
print("       🗄️  DATABASE SUMMARY STATS")
print("="*45)
stats = {
    "total_violations" : conn.execute("SELECT COUNT(*) FROM violations").fetchone()[0],
    "total_accidents"  : conn.execute("SELECT COUNT(*) FROM violations WHERE Accident=1").fetchone()[0],
    "total_fatal"      : conn.execute("SELECT COUNT(*) FROM violations WHERE Fatal=1").fetchone()[0],
    "total_alcohol"    : conn.execute("SELECT COUNT(*) FROM violations WHERE Alcohol=1").fetchone()[0],
    "total_hazmat"     : conn.execute("SELECT COUNT(*) FROM violations WHERE HAZMAT=1").fetchone()[0],
}
for k, v in stats.items():
    print(f"  {k:<22}: {v:,}")
print("="*45)

conn.close()

Tables in DB: ['violations']
✅ Indexes created!

📊 Query 1 — Top Violation Types:
ViolationCategory  total   pct
            Other 979699 47.33
         Speeding 360764 17.43
     Registration 306490 14.81
          License 212308 10.26
      DUI/Alcohol  72820  3.52
   Traffic Signal  71718  3.46
         Seatbelt  35080  1.69
        Equipment  27823  1.34
        Insurance   3413  0.16

🚨 Query 2 — Districts with Highest Accident Rates:
                                      SubAgency  total_stops  accidents  accident_rate
                          4th District, Wheaton       449193      16736           3.73
                         2nd District, Bethesda       325456      10487           3.22
                    3rd District, Silver Spring       373654      10607           2.84
                       5th District, Germantown       245758       5789           2.36
                        1st District, Rockville       240901       5393           2.24
6th District, Gaithersburg / Montg